In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns",None)

In [5]:
DATA_PATH = "../data/raw/normalized_blocker.csv"
df = pd.read_csv(DATA_PATH)
df.head()

,blocker_id,source_type,source_id,team_id,sprint_id,date_logged,category,description,is_external_dependency,resolution_time_days,status
0,BLK-1001,Jira,SRC-5001,TEAM-GAMMA,SPRINT-006,2026-04-07,Cross-Team Dependency,Impediment related to cross-team dependency im...,True,1,Resolved
1,BLK-1002,Slack,SRC-5002,TEAM-BETA,SPRINT-006,2026-04-08,CI/CD & Pipeline,Impediment related to ci/cd & pipeline impacti...,False,3,Resolved
2,BLK-1003,Slack,SRC-5003,TEAM-GAMMA,SPRINT-005,2026-04-09,Environment & Access,Impediment related to environment & access imp...,True,5,Resolved
3,BLK-1004,Slack,SRC-5004,TEAM-GAMMA,SPRINT-006,2026-04-10,Environment & Access,Impediment related to environment & access imp...,True,8,Resolved
4,BLK-1005,CSV,SRC-5005,TEAM-ALPHA,SPRINT-005,2026-04-11,Cross-Team Dependency,Impediment related to cross-team dependency im...,True,2,Resolved


In [6]:
df.dtypes

blocker_id                  str
source_type                 str
source_id                   str
team_id                     str
sprint_id                   str
date_logged                 str
category                    str
description                 str
is_external_dependency     bool
resolution_time_days      int64
status                      str
dtype: object

In [7]:
def convert_dates(df, columns, date_format):

    """
    Converts string dates into pandas datetime.

    Input:
    DataFrame containing string date columns

    Output:
    DataFrame with datetime columns

    Assumption:
    All dates follow the provided format.
    """

    df = df.copy()


    for col in columns:

        print(f"Before: {col} -> {df[col].dtype}")


        df[col] = pd.to_datetime(
            df[col],
            format=date_format
        )


        print(f"After : {col} -> {df[col].dtype}")
        print()


    return df

In [8]:
typed_df = convert_dates(
    df,
    [
        "date_logged"
    ],
    "%Y-%m-%d"
)

Before: date_logged -> str
After : date_logged -> datetime64[us]



In [9]:
def convert_boolean(df, columns):

    """
    Converts binary columns into boolean datatype.

    Accepted values:
    True/False
    1/0
    yes/no

    Output:
    Boolean columns
    """

    df=df.copy()


    mapping={
        1:True,
        0:False,
        "1":True,
        "0":False,
        "yes":True,
        "no":False,
        "true":True,
        "false":False
    }


    for col in columns:

        print(
            col,
            "before:",
            df[col].unique()
        )


        df[col]=(
            df[col]
            .map(mapping)
            .astype(bool)
        )


        print(
            col,
            "after:",
            df[col].unique()
        )


    return df

In [10]:
typed_df = convert_boolean(
    typed_df,
    [
        "is_external_dependency"
    ]
)

is_external_dependency before: [ True False]
is_external_dependency after: [ True False]


In [11]:
def convert_currency(df, columns):

    """
    Removes currency symbols and converts values to float.

    Example:

    "$150.50" --> 150.50

    """

    df=df.copy()


    for col in columns:


        df[col]=(
            df[col]
            .astype(str)
            .str.replace("$","",regex=False)
            .str.replace(",","",regex=False)
        )


        df[col]=pd.to_numeric(
            df[col],
            errors="coerce"
        )


        print(
            f"{col} converted to float"
        )


    return df

In [12]:
def compare_types(before, after):


    report=pd.DataFrame({

        "column":before.columns,

        "before_dtype":
            before.dtypes.values,

        "after_dtype":
            after.dtypes.values,

        "changed":
            (
                before.dtypes.values !=
                after.dtypes.values
            )

    })


    display(report)


    Path("../output").mkdir(
        exist_ok=True
    )


    report.to_csv(
        "../output/dtype_conversion_report.csv",
        index=False
    )


    return report

In [13]:
report = compare_types(
    df,
    typed_df
)

,column,before_dtype,after_dtype,changed
0,blocker_id,str,str,False
1,source_type,str,str,False
2,source_id,str,str,False
3,team_id,str,str,False
4,sprint_id,str,str,False
5,date_logged,str,datetime64[us],True
6,category,str,str,False
7,description,str,str,False
8,is_external_dependency,bool,bool,False
9,resolution_time_days,int64,int64,False


In [14]:
Path("../data/processed").mkdir(
    exist_ok=True
)


typed_df.to_csv(
    "../data/processed/typed_blockers.csv",
    index=False
)

print("Typed dataset saved")

Typed dataset saved
